# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [25]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [26]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [27]:
spaceship = spaceship.dropna()

In [28]:
spaceship["Cabin"] = spaceship["Cabin"].astype(str).str[0]

In [29]:
spaceship = spaceship.drop(columns=["PassengerId", "Name"])


In [30]:
categorical_cols = ["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP"]
spaceship = pd.get_dummies(spaceship, columns=categorical_cols, drop_first=True)

In [31]:
X = spaceship.drop("Transported", axis=1)
y = spaceship["Transported"]

In [32]:
X.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,True,False,False,True,False,False,False,False,False,False,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,False,False,False,False,False,False,False,True,False,False,False,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,True,False,False,False,False,False,False,False,False,False,False,True,True
3,33.0,0.0,1283.0,371.0,3329.0,193.0,True,False,False,False,False,False,False,False,False,False,False,True,False
4,16.0,303.0,70.0,151.0,565.0,2.0,False,False,False,False,False,False,False,True,False,False,False,True,False


**Perform Train Test Split**

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [34]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=200,
    max_samples=1.0,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

bag_clf.fit(X_train, y_train)
bagging_acc = bag_clf.score(X_test, y_test)
print("Bagging accuracy:", bagging_acc)

Bagging accuracy: 0.8040847201210287


- Random Forests

In [35]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)

rf_clf.fit(X_train, y_train)
rf_acc = rf_clf.score(X_test, y_test)
print("Random Forest accuracy:", rf_acc)

Random Forest accuracy: 0.8078668683812406


- Gradient Boosting

In [36]:
from sklearn.ensemble import GradientBoostingClassifier

gb_clf = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_clf.fit(X_train, y_train)
gb_acc = gb_clf.score(X_test, y_test)
print("Gradient Boosting accuracy:", gb_acc)

Gradient Boosting accuracy: 0.8063540090771558


- Adaptive Boosting

In [37]:
from sklearn.ensemble import AdaBoostClassifier

ada_clf = AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)

ada_clf.fit(X_train, y_train)
ada_acc = ada_clf.score(X_test, y_test)
print("AdaBoost accuracy:", ada_acc)

AdaBoost accuracy: 0.7813918305597579


Which model is the best and why?

In [38]:
print("---- MODEL COMPARISON ----")
print("Bagging accuracy:        ", bagging_acc)
print("Random Forest accuracy:  ", rf_acc)
print("Gradient Boosting acc.:  ", gb_acc)
print("AdaBoost accuracy:       ", ada_acc)

---- MODEL COMPARISON ----
Bagging accuracy:         0.8040847201210287
Random Forest accuracy:   0.8078668683812406
Gradient Boosting acc.:   0.8063540090771558
AdaBoost accuracy:        0.7813918305597579


In [39]:
# The best model is the one with the highest test accuracy.
# Usually Gradient Boosting or Random Forest perform best.
# Random Forest reduces variance by combining many decorrelated trees.
# Gradient Boosting reduces bias by learning sequentially and correcting previous mistakes.
# AdaBoost is more sensitive to noise but can perform well with tuned learning_rate.
# Bagging improves stability but lacks the sequential learning advantages of boosting methods.
